# RAW DATA INGESTION
## Purpose

This notebook loads raw CSV files from S3 into Bronze Delta tables.

Each `COPY INTO` statement ingests only new files that have not already been loaded into the target table, making the ingestion notebook safe to rerun.

The Bronze layer keeps source data close to its raw form and adds ingestion metadata:
- `_source_file` identifies the source CSV file.
- `_ingested_at` records when the file was loaded into Databricks.

## Devices

Loads raw device reference data from S3 into `bronze.devices`.

This table stores device attributes such as device type, zone, and operational status. No cleaning or validation is applied at this stage.

In [0]:
COPY INTO parcel_sorting_hub.bronze.devices
FROM (
  SELECT 
    device_id,
    type,
    zone,
    status,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/devices/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Exceptions

Loads raw parcel exception records into `bronze.exceptions`.

These records capture operational issues reported for parcels, including error codes, device references, status, and resolution timestamps.

In [0]:
COPY INTO parcel_sorting_hub.bronze.exceptions
FROM (
  SELECT 
    exception_id,
    parcel_id,   
    device_id,   
    error_code,  
    status,      
    reported_at, 
    resolved_at, 
    employee_id,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/exceptions/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Intake

Loads raw delivery intake records into `bronze.intake`.

This table captures inbound delivery events, including dock assignment, unload timestamps, parcel counts, truck plate, and source hub.

In [0]:
COPY INTO parcel_sorting_hub.bronze.intake
FROM (
  SELECT 
    delivery_id,  
    dock_id,      
    arrival_time, 
    unload_start, 
    unload_end,   
    parcel_count, 
    truck_plate,  
    source_hub_id,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/intake/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Loading

Loads raw loading records into `bronze.loading`.

This table captures outbound loading activity, including dock, route, employee, start time, close time, and parcel count.

In [0]:
COPY INTO parcel_sorting_hub.bronze.loading
FROM (
  SELECT 
    loading_id,
    dock_id,
    start_time,
    close_time,
    parcel_count,
    route_id,
    employee_id,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/loading/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Parcels

Loads raw parcel master records into `bronze.parcels`.

This table contains parcel-level attributes such as dimensions, weight, service type, status, hub references, delivery ID, and loading ID.

In [0]:
COPY INTO parcel_sorting_hub.bronze.parcels
FROM (
  SELECT 
    parcel_id,         
    status,            
    weight_kg,        
    length_cm,         
    width_cm,          
    height_cm,         
    service_type,      
    received_at,       
    source_hub_id,     
    delivery_id,       
    destination_hub_id,
    loading_id,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/parcels/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Scans

Loads raw parcel scan events into `bronze.scans`.

This table captures scan-level events from devices, including scan type, result, timestamp, and dynamic weight.

In [0]:
COPY INTO parcel_sorting_hub.bronze.scans
FROM (
  SELECT 
    scan_id,       
    parcel_id,     
    device_id,     
    scan_type,     
    result,        
    scan_timestamp,
    dynamic_weight,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/scans/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Sorting

Loads raw sorting events into `bronze.sorting`.

This table captures parcel movement through sorters and chutes, including event time and sorting result.

In [0]:
COPY INTO parcel_sorting_hub.bronze.sorting
FROM (
  SELECT 
    event_id,  
    parcel_id, 
    sorter_id, 
    chute_id,  
    entry_time,
    result,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/sorting/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

## Status History

Loads raw parcel status changes into `bronze.status_history`.

This table records historical parcel status transitions and the timestamp of each change.

In [0]:
COPY INTO parcel_sorting_hub.bronze.status_history
FROM (
  SELECT 
    history_id,      
    parcel_id,       
    status_name,          
    change_timestamp,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/status_history/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

In [0]:
COPY INTO parcel_sorting_hub.bronze.employees
FROM (
  SELECT 
    employee_id,      
    full_name,       
    date_of_birth,          
    date_joined,
    _metadata.file_name AS _source_file,
    current_timestamp() AS _ingested_at
  FROM 's3://sortownia-data/parcel_sorting_hub/data-raw-daily-batch/employees/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');